# Module 15: Error Handling and Debugging

## Writing Robust Bioinformatics Code

---

### Learning Objectives
- Master `try`/`except`/`else`/`finally` for structured error handling
- Raise exceptions and create custom exception classes
- Understand the Python exception hierarchy
- Debug effectively with print debugging, `assert`, and `pdb`/`breakpoint()`
- Use the `logging` module for production code
- Handle real-world bioinformatics errors: malformed FASTA, invalid sequences, missing GFF data

---

## Why error handling matters in bioinformatics

Bioinformatics data is notoriously messy: FASTA files with empty sequences, VCF lines with the wrong number of fields, gene IDs that don't match between databases, quality scores outside valid range. A pipeline that crashes on the first bad record and gives a cryptic traceback wastes hours. Good error handling means clear messages, graceful batch processing, and auditable logs.

## How to use this notebook

Run cells from top to bottom on the first pass — later cells depend on functions and variables defined earlier. Once you have run through the notebook, feel free to modify parameters and re-run individual sections.

Each section has runnable examples first, followed by exercises. Try the exercise before looking at the solution cell below it.

## Complicated moments explained

**1. Always catch specific exceptions**
`except:` catches *everything* including `KeyboardInterrupt` and `SystemExit`, making it impossible to stop the script with Ctrl+C. Always name the exception: `except ValueError`, `except FileNotFoundError`.

**2. `else` on a `try` block**
The `else` clause runs only if the `try` block raised *no* exception. Use it to separate "code that might fail" from "code that should only run on success":
```python
try:
    seq = parse_fasta(file)
except ParseError:
    log.warning(f"bad file: {file}")
else:
    analyze(seq)   # only runs if parse succeeded
```

**3. `raise ... from ...` preserves the chain**
When catching one exception and raising another, use `raise NewError(...) from original` to keep the full traceback chain. Without `from`, the original cause is hidden.

**4. `assert` is for development invariants only**
Python disables `assert` statements when run with `python -O`. Never use `assert` to validate external data. Use `if not condition: raise ValueError(...)` instead.

In [ ]:
def gc_content(seq):
    """Calculate GC content with basic error handling."""
    try:
        seq = seq.upper()
        gc = seq.count('G') + seq.count('C')
        return gc / len(seq) * 100
    except ZeroDivisionError:
        print("Warning: empty sequence")
        return 0.0
    except AttributeError:
        print(f"Warning: expected string, got {type(seq).__name__}")
        return None


print(f"Normal: {gc_content('ATGCGC'):.1f}%")
print(f"Empty:  {gc_content('')}%")
print(f"Wrong type: {gc_content(12345)}")

**Rule:** Always catch specific exceptions. A bare `except:` hides bugs
by catching everything, including `KeyboardInterrupt` and `SystemExit`.

## 3. The Complete `try`/`except`/`else`/`finally` Block

- `try` -- code that might raise an exception
- `except` -- handle specific exceptions
- `else` -- runs only if **no** exception occurred (the "happy path")
- `finally` -- **always** runs (cleanup code)

In [ ]:
def read_fasta(filename):
    """Read a FASTA file with full error handling."""
    sequences = {}
    current_id = None
    file = None

    try:
        file = open(filename, 'r')

    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")
        return {}

    except PermissionError:
        print(f"Error: No permission to read '{filename}'")
        return {}

    else:
        # Only runs if the file opened successfully
        for line in file:
            line = line.strip()
            if line.startswith('>'):
                current_id = line[1:].split()[0]
                sequences[current_id] = []
            elif current_id:
                sequences[current_id].append(line)
        sequences = {k: ''.join(v) for k, v in sequences.items()}
        print(f"Read {len(sequences)} sequences from {filename}")

    finally:
        # Always close the file if it was opened
        if file:
            file.close()
            print("File closed")

    return sequences


# Create a test file
with open('test.fasta', 'w') as f:
    f.write(">seq1 test sequence\nATGCATGC\n>seq2 another\nGGCCAATT\n")

print("--- Existing file ---")
seqs = read_fasta('test.fasta')
print(f"Result: {seqs}\n")

print("--- Non-existent file ---")
seqs = read_fasta('does_not_exist.fasta')

## 4. Raising Exceptions

Use `raise` when your code detects an error condition. Provide a clear message
that helps the caller understand what went wrong.

In [ ]:
def validate_dna(sequence, min_length=3):
    """Validate a DNA sequence, raising clear errors on problems."""
    if not isinstance(sequence, str):
        raise TypeError(f"Expected string, got {type(sequence).__name__}")

    if len(sequence) < min_length:
        raise ValueError(
            f"Sequence too short: {len(sequence)} bp (minimum {min_length})"
        )

    invalid = set(sequence.upper()) - set('ATGCN')
    if invalid:
        raise ValueError(f"Invalid DNA characters: {invalid}")

    return sequence.upper()


test_cases = [
    ("ATGCATGC", "Valid DNA"),
    ("AT", "Too short"),
    ("ATGXYZ", "Invalid characters"),
    (42, "Wrong type"),
]

for seq, description in test_cases:
    try:
        result = validate_dna(seq)
        print(f"  {description}: OK -> {result}")
    except (TypeError, ValueError) as e:
        print(f"  {description}: FAILED -> {e}")

## 5. Custom Exceptions

Custom exceptions make your code's errors self-documenting and allow
callers to catch specific categories of errors.

In [ ]:
class BioinformaticsError(Exception):
    """Base exception for all bioinformatics errors."""
    pass


class InvalidSequenceError(BioinformaticsError):
    """Raised when a sequence contains invalid characters."""
    def __init__(self, invalid_chars, seq_type="DNA"):
        self.invalid_chars = invalid_chars
        self.seq_type = seq_type
        super().__init__(f"Invalid {seq_type} characters: {invalid_chars}")


class SequenceLengthError(BioinformaticsError):
    """Raised when a sequence does not meet length requirements."""
    def __init__(self, actual, minimum=None, maximum=None):
        self.actual = actual
        if minimum and actual < minimum:
            msg = f"Sequence too short: {actual} < {minimum}"
        elif maximum and actual > maximum:
            msg = f"Sequence too long: {actual} > {maximum}"
        else:
            msg = f"Invalid sequence length: {actual}"
        super().__init__(msg)


class FastaParseError(BioinformaticsError):
    """Raised on FASTA format violations, with context."""
    def __init__(self, message, filename=None, line_number=None, line_content=None):
        self.filename = filename
        self.line_number = line_number
        self.line_content = line_content
        parts = [message]
        if filename:
            parts.append(f"file={filename}")
        if line_number:
            parts.append(f"line={line_number}")
        if line_content:
            parts.append(f"content='{line_content[:50]}'")
        super().__init__(" | ".join(parts))


class TranslationError(BioinformaticsError):
    """Raised when DNA/RNA translation fails."""
    pass

In [ ]:
# Using custom exceptions
def validate_and_translate(sequence):
    """Validate DNA and translate to protein."""
    seq = sequence.upper()

    # Check for invalid characters
    invalid = set(seq) - set('ATGC')
    if invalid:
        raise InvalidSequenceError(invalid, seq_type="DNA")

    # Check length
    if len(seq) < 3:
        raise SequenceLengthError(len(seq), minimum=3)

    if len(seq) % 3 != 0:
        raise TranslationError(
            f"Sequence length {len(seq)} is not divisible by 3"
        )

    # Simple translation
    codon_table = {
        'ATG': 'M', 'GCT': 'A', 'GCC': 'A', 'TAA': '*', 'TAG': '*', 'TGA': '*',
        'TTT': 'F', 'TTC': 'F', 'TGG': 'W', 'GAA': 'E', 'GAG': 'E',
    }
    protein = []
    for i in range(0, len(seq), 3):
        codon = seq[i:i+3]
        aa = codon_table.get(codon, 'X')
        if aa == '*':
            break
        protein.append(aa)
    return ''.join(protein)


# Test various error conditions
test_cases = [
    "ATGGCTTAG",     # valid
    "ATGXYZ",        # invalid characters
    "AT",            # too short
    "ATGGC",         # not divisible by 3
]

for seq in test_cases:
    try:
        protein = validate_and_translate(seq)
        print(f"  '{seq}' -> '{protein}'")
    except InvalidSequenceError as e:
        print(f"  '{seq}' -> Invalid: {e}")
    except SequenceLengthError as e:
        print(f"  '{seq}' -> Length: {e}")
    except TranslationError as e:
        print(f"  '{seq}' -> Translation: {e}")
    except BioinformaticsError as e:
        # Catches any BioinformaticsError subclass not caught above
        print(f"  '{seq}' -> General bio error: {e}")

## 6. Exception Chaining

Use `raise ... from ...` to preserve the original error context.
This is essential for debugging multi-layered pipelines.

In [ ]:
def parse_gff_line(line, line_number=None):
    """Parse a single GFF3 line into a dict."""
    fields = line.strip().split('\t')
    if len(fields) != 9:
        raise FastaParseError(
            f"Expected 9 tab-separated fields, got {len(fields)}",
            line_number=line_number,
            line_content=line.strip()
        )

    try:
        return {
            'seqid': fields[0],
            'source': fields[1],
            'type': fields[2],
            'start': int(fields[3]),
            'end': int(fields[4]),
            'score': fields[5],
            'strand': fields[6],
            'phase': fields[7],
            'attributes': fields[8],
        }
    except ValueError as e:
        # Chain the original ValueError into our custom exception
        raise FastaParseError(
            f"Cannot parse coordinates",
            line_number=line_number,
            line_content=line.strip()
        ) from e


# Valid GFF line
good_line = "chr1\tRefSeq\tgene\t11874\t14409\t.\t+\t.\tID=gene1;Name=DDX11L1"
print(parse_gff_line(good_line))

# Bad coordinate
bad_line = "chr1\tRefSeq\tgene\tNOT_A_NUMBER\t14409\t.\t+\t.\tID=gene1"
try:
    parse_gff_line(bad_line, line_number=42)
except FastaParseError as e:
    print(f"\nParse error: {e}")
    if e.__cause__:
        print(f"  Caused by: {e.__cause__}")

## 7. Handling Malformed FASTA Files

A strict FASTA parser that gives useful error messages.

In [ ]:
def strict_fasta_parser(filename):
    """Parse FASTA with strict validation and detailed errors."""
    sequences = {}
    current_id = None
    current_seq = []

    with open(filename, 'r') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            if line.startswith('>'):
                # Save previous record
                if current_id:
                    seq = ''.join(current_seq)
                    if not seq:
                        raise FastaParseError(
                            "Empty sequence",
                            filename, line_num, f">{current_id}"
                        )
                    sequences[current_id] = seq

                # Parse header
                if len(line) == 1:
                    raise FastaParseError(
                        "Empty sequence ID (header is just '>')",
                        filename, line_num, line
                    )
                current_id = line[1:].split()[0]
                current_seq = []

            else:
                if current_id is None:
                    raise FastaParseError(
                        "Sequence data before first header",
                        filename, line_num, line
                    )
                # Validate sequence characters
                invalid = set(line.upper()) - set('ATGCNRYSWKMBDHV.-')
                if invalid:
                    raise FastaParseError(
                        f"Invalid characters: {invalid}",
                        filename, line_num, line
                    )
                current_seq.append(line.upper())

    # Save last record
    if current_id:
        seq = ''.join(current_seq)
        if not seq:
            raise FastaParseError("Empty sequence for last record", filename)
        sequences[current_id] = seq

    return sequences


# Test with a malformed file
with open('bad.fasta', 'w') as f:
    f.write(">seq1\nATGC\n>\nGGCC\n")  # second header is empty

try:
    seqs = strict_fasta_parser('bad.fasta')
except FastaParseError as e:
    print(f"Parse error: {e}")

# Test with data before header
with open('bad2.fasta', 'w') as f:
    f.write("ATGCATGC\n>seq1\nGGCC\n")  # sequence before header

try:
    seqs = strict_fasta_parser('bad2.fasta')
except FastaParseError as e:
    print(f"Parse error: {e}")

## 8. Graceful Degradation: Processing Batches with Errors

When processing many sequences, you often want to skip bad ones and
continue rather than crash the entire pipeline.

In [ ]:
def batch_gc_analysis(sequences, strict=False):
    """
    Calculate GC content for a batch of sequences.

    strict=True:  raise on first error
    strict=False: skip bad sequences, report errors at the end
    """
    results = {}
    errors = []

    for seq_id, seq in sequences.items():
        try:
            # Validate
            if not seq:
                raise SequenceLengthError(0, minimum=1)
            invalid = set(seq.upper()) - set('ATGCN')
            if invalid:
                raise InvalidSequenceError(invalid)

            # Analyze
            s = seq.upper()
            gc = (s.count('G') + s.count('C')) / len(s) * 100
            results[seq_id] = {'gc': round(gc, 2), 'length': len(s)}

        except BioinformaticsError as e:
            if strict:
                raise
            errors.append((seq_id, str(e)))

    # Report
    print(f"Processed: {len(results)}/{len(sequences)} successful")
    if errors:
        print(f"Errors ({len(errors)}):")
        for seq_id, err in errors:
            print(f"  {seq_id}: {err}")

    return results


test_batch = {
    'gene1': 'ATGCATGCATGC',
    'gene2': 'GGCCAATTGGCC',
    'gene3': 'ATGXYZ',        # invalid chars
    'gene4': '',               # empty
    'gene5': 'TTAACCGG',
}

print("=== Lenient mode ===")
results = batch_gc_analysis(test_batch, strict=False)
print(f"Results: {results}\n")

print("=== Strict mode ===")
try:
    results = batch_gc_analysis(test_batch, strict=True)
except BioinformaticsError as e:
    print(f"Stopped at error: {e}")

## 9. Common Exceptions in Bioinformatics Code

Let us look at the most common exceptions you will encounter and how to handle them.

In [ ]:
# KeyError: accessing missing dictionary keys (common with parsed data)
gene_info = {'BRCA1': {'chr': 17, 'start': 43044295}}

try:
    tp53_start = gene_info['TP53']['start']
except KeyError as e:
    print(f"KeyError: gene {e} not found in database")


# IndexError: accessing beyond list bounds (common when parsing fixed-format lines)
gff_fields = "chr1\tRefSeq\tgene".split('\t')  # only 3 fields instead of 9

try:
    strand = gff_fields[6]
except IndexError:
    print(f"IndexError: GFF line has only {len(gff_fields)} fields (expected 9)")


# ValueError: wrong format (common when converting coordinates)
try:
    start = int("12,345")  # comma in number
except ValueError as e:
    print(f"ValueError: {e}")


# UnicodeDecodeError: binary data read as text
try:
    with open('test.fasta', 'r', encoding='ascii') as f:
        content = f.read()
    print("File read OK")
except UnicodeDecodeError as e:
    print(f"Encoding error: {e}")

## 10. Assertions: Development-Time Checks

`assert` statements check invariants during development. They can be
disabled globally with `python -O`, so never use them for input validation
in production code.

In [ ]:
def translate_codon(codon):
    """Translate a single codon, with development-time assertions."""
    # Preconditions
    assert isinstance(codon, str), f"codon must be str, got {type(codon).__name__}"
    assert len(codon) == 3, f"codon must be 3 chars, got {len(codon)}"

    codon = codon.upper()
    assert set(codon) <= set('ATGC'), f"invalid bases in codon: {codon}"

    table = {
        'ATG': 'M', 'TGG': 'W', 'TTT': 'F', 'TTC': 'F',
        'GCT': 'A', 'GCC': 'A', 'GCA': 'A', 'GCG': 'A',
        'TAA': '*', 'TAG': '*', 'TGA': '*',
    }
    result = table.get(codon, 'X')

    # Postcondition
    assert result in 'ACDEFGHIKLMNPQRSTVWYX*', f"unexpected result: {result}"
    return result


print(translate_codon('ATG'))  # M

try:
    translate_codon('AT')  # too short
except AssertionError as e:
    print(f"Assertion failed: {e}")

try:
    translate_codon('XYZ')  # invalid bases
except AssertionError as e:
    print(f"Assertion failed: {e}")

## 11. Print Debugging with Toggle

Print debugging is simple but effective. Using a flag or a helper
function makes it easy to turn on/off.

In [ ]:
DEBUG = True


def debug(*args, **kwargs):
    """Conditional debug output."""
    if DEBUG:
        print("[DEBUG]", *args, **kwargs)


def find_orfs(sequence, min_length=30):
    """Find open reading frames with debug output."""
    orfs = []
    sequence = sequence.upper()
    debug(f"Searching {len(sequence)} bp sequence")

    for frame in range(3):
        debug(f"  Frame +{frame}")
        in_orf = False
        start_pos = None

        for i in range(frame, len(sequence) - 2, 3):
            codon = sequence[i:i+3]
            if codon == 'ATG' and not in_orf:
                in_orf = True
                start_pos = i
                debug(f"    Start codon at position {i}")
            elif codon in ('TAA', 'TAG', 'TGA') and in_orf:
                length = i + 3 - start_pos
                debug(f"    Stop codon at {i}, ORF length = {length} bp")
                if length >= min_length:
                    orfs.append({'start': start_pos, 'end': i + 3, 'length': length})
                in_orf = False

    debug(f"Found {len(orfs)} ORFs >= {min_length} bp")
    return orfs


orfs = find_orfs("ATGAAAGCTTAATGCCCGCTTGAATGAAACCC", min_length=9)
print(f"\nResult: {orfs}")

## 12. Using `pdb` / `breakpoint()`

Python's built-in debugger lets you pause execution, inspect variables,
and step through code interactively.

Common `pdb` commands:
- `n` (next) -- execute next line
- `s` (step) -- step into function call
- `c` (continue) -- run until next breakpoint
- `p variable` -- print a variable
- `l` (list) -- show surrounding code
- `q` (quit) -- exit debugger

In [ ]:
def mysterious_gc_bug(sequences):
    """This function has a subtle bug. Use breakpoint() to find it."""
    results = {}

    for name, seq in sequences.items():
        # Uncomment the next line to debug:
        # breakpoint()

        gc_count = seq.count('G') + seq.count('c')  # Bug: lowercase 'c'
        total = len(seq)
        results[name] = gc_count / total * 100

    return results


test = {'seq1': 'ATGCATGC', 'seq2': 'GGCCAATT'}
print("Results (buggy):")
print(mysterious_gc_bug(test))

# Expected: seq1 = 50%, seq2 = 50%
# Can you spot the bug without running the debugger?

## 13. The `logging` Module

For production code, `logging` is better than print statements because:
- Log levels (DEBUG, INFO, WARNING, ERROR, CRITICAL) let you filter output
- Logs can go to files, not just the console
- Timestamps and context are automatic

In [ ]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%H:%M:%S'
)

logger = logging.getLogger('bioinfo_pipeline')

In [ ]:
def process_fasta_with_logging(filename):
    """Process a FASTA file with proper logging."""
    logger.info(f"Starting to process {filename}")

    try:
        sequences = read_fasta(filename)
    except FileNotFoundError:
        logger.error(f"File not found: {filename}")
        return {}

    results = {}
    for seq_id, seq in sequences.items():
        logger.debug(f"Processing {seq_id} ({len(seq)} bp)")

        if not seq:
            logger.warning(f"Skipping empty sequence: {seq_id}")
            continue

        gc = (seq.upper().count('G') + seq.upper().count('C')) / len(seq) * 100
        results[seq_id] = round(gc, 2)
        logger.info(f"{seq_id}: {len(seq)} bp, GC={gc:.1f}%")

    logger.info(f"Completed: {len(results)} sequences processed")
    return results


results = process_fasta_with_logging('test.fasta')

### Log Levels

| Level | When to use |
|-------|-------------|
| `DEBUG` | Detailed diagnostic info (variable values, loop progress) |
| `INFO` | Confirmation that things are working (started, completed) |
| `WARNING` | Something unexpected but not fatal (skipped record) |
| `ERROR` | A serious problem (file not found, parse failure) |
| `CRITICAL` | Program cannot continue (out of memory, corrupt database) |

## 14. EAFP vs LBYL

Python favors **EAFP** (Easier to Ask Forgiveness than Permission):
try the operation and handle the error, rather than checking in advance.

In [ ]:
gene_db = {
    'BRCA1': {'sequence': 'ATGGATTTCG', 'chromosome': 17},
    'TP53':  {'sequence': 'ATGGAGGAGC', 'chromosome': 17},
}


# LBYL (Look Before You Leap) -- check first, then act
def get_gene_lbyl(name):
    if name in gene_db:
        if 'sequence' in gene_db[name]:
            return gene_db[name]['sequence']
    return None


# EAFP (Easier to Ask Forgiveness than Permission) -- try it, handle errors
def get_gene_eafp(name):
    try:
        return gene_db[name]['sequence']
    except KeyError:
        return None


# Both work, but EAFP is more Pythonic and often faster
print(f"LBYL: {get_gene_lbyl('BRCA1')}")
print(f"EAFP: {get_gene_eafp('BRCA1')}")
print(f"Missing (LBYL): {get_gene_lbyl('EGFR')}")
print(f"Missing (EAFP): {get_gene_eafp('EGFR')}")

## 15. Handling Missing Data in GFF Files

In [ ]:
def parse_gff_attributes(attr_string):
    """Parse GFF3 attribute string like 'ID=gene1;Name=BRCA1;biotype=protein_coding'."""
    attrs = {}
    for pair in attr_string.split(';'):
        pair = pair.strip()
        if '=' in pair:
            key, value = pair.split('=', 1)
            attrs[key] = value
    return attrs


def safe_parse_gff(lines):
    """Parse GFF lines, handling missing/malformed data gracefully."""
    records = []
    errors = []

    for i, line in enumerate(lines, 1):
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        fields = line.split('\t')

        # Check field count
        if len(fields) < 9:
            errors.append((i, f"Only {len(fields)} fields (need 9)"))
            continue

        # Parse coordinates safely
        try:
            start = int(fields[3])
            end = int(fields[4])
        except ValueError:
            errors.append((i, f"Bad coordinates: '{fields[3]}', '{fields[4]}'"))
            continue

        # Parse attributes
        attrs = parse_gff_attributes(fields[8])
        name = attrs.get('Name', attrs.get('ID', 'unknown'))

        records.append({
            'seqid': fields[0],
            'type': fields[2],
            'start': start,
            'end': end,
            'strand': fields[6],
            'name': name,
        })

    if errors:
        print(f"Warnings: {len(errors)} lines skipped")
        for line_num, msg in errors[:5]:
            print(f"  Line {line_num}: {msg}")

    return records


gff_lines = [
    "# GFF3 example",
    "chr17\tRefSeq\tgene\t43044295\t43170245\t.\t-\t.\tID=BRCA1;Name=BRCA1",
    "chr17\tRefSeq\tgene\t7668402\t7687538\t.\t-\t.\tID=TP53;Name=TP53",
    "chr7\tbad_line_only_3_fields",  # malformed
    "chr7\tRefSeq\tgene\tNOPE\t55211628\t.\t+\t.\tID=EGFR",  # bad coord
    "chr12\tRefSeq\texon\t25205246\t25209911\t.\t-\t.\tID=KRAS;Name=KRAS",
]

records = safe_parse_gff(gff_lines)
print(f"\nParsed {len(records)} records:")
for r in records:
    print(f"  {r['name']}: {r['seqid']}:{r['start']}-{r['end']} ({r['strand']})")

---

## Exercises

### Exercise 1: Robust Sequence File Reader

Write a function `read_sequences(filename)` that:
- Handles `FileNotFoundError` with a clear message
- Detects whether the file is FASTA (starts with '>') or plain text (one sequence per line)
- Returns a list of sequences
- Skips empty lines
- Raises `FastaParseError` if a FASTA file has a malformed header

In [ ]:
# Exercise 1: Your solution

# Test:
# Create test files and verify:
# seqs = read_sequences("test.fasta")    # FASTA format
# seqs = read_sequences("seqs.txt")      # plain text
# seqs = read_sequences("missing.txt")   # FileNotFoundError

### Exercise 2: Custom Exception Hierarchy for a Pipeline

Design an exception hierarchy for a variant calling pipeline:
- `PipelineError` (base)
  - `InputFormatError` -- bad VCF/BAM format
  - `QualityError` -- quality below threshold
  - `ReferenceError` -- chromosome not in reference genome

Write a function `validate_variant(chrom, pos, ref, alt, quality)` that
raises the appropriate exception for each error condition.

In [ ]:
# Exercise 2: Your solution

# Test:
# validate_variant("chr1", 12345, "A", "G", 30)     # OK
# validate_variant("chrZ", 12345, "A", "G", 30)     # ReferenceError
# validate_variant("chr1", 12345, "A", "G", 5)      # QualityError
# validate_variant("chr1", -1, "A", "G", 30)        # InputFormatError

### Exercise 3: Logging-Based Sequence Processor

Write a function `analyze_fasta_logged(filename)` that:
- Uses `logging` (not print) for all output
- Logs INFO for each sequence processed
- Logs WARNING for sequences with ambiguous bases (N)
- Logs ERROR for unparseable lines
- Returns a dict of `{seq_id: gc_content}`

In [ ]:
# Exercise 3: Your solution

# Test:
# results = analyze_fasta_logged("test.fasta")

### Exercise 4: Safe Dictionary Access for Parsed Data

Write a function `safe_get_nested(data, *keys, default=None)` that safely
accesses nested dictionaries (common when working with JSON from APIs).
Return `default` if any key is missing.

Example: `safe_get_nested(data, 'result', 'sequences', 0, 'id')` should
work even if any intermediate key does not exist.

In [ ]:
# Exercise 4: Your solution

# Test:
# api_response = {
#     'result': {
#         'sequences': [
#             {'id': 'NM_007294', 'organism': 'Homo sapiens'},
#         ]
#     }
# }
# print(safe_get_nested(api_response, 'result', 'sequences', 0, 'id'))  # NM_007294
# print(safe_get_nested(api_response, 'result', 'missing', 'key'))      # None
# print(safe_get_nested(api_response, 'nope', default='N/A'))           # N/A

### Exercise 5: Retry with Exponential Backoff

Write a decorator `retry_with_backoff(max_attempts=3, initial_delay=1.0)`
that retries on failure, doubling the delay between each attempt
(1s, 2s, 4s, ...).

Test it with a function that simulates an unreliable NCBI query.

In [ ]:
# Exercise 5: Your solution

# Test:
# @retry_with_backoff(max_attempts=4, initial_delay=0.1)
# def query_ncbi(accession):
#     import random
#     if random.random() < 0.7:
#         raise ConnectionError(f"Timeout querying {accession}")
#     return {"id": accession, "sequence": "ATGC" * 100}
#
# try:
#     result = query_ncbi("NM_007294")
#     print(f"Got result for {result['id']}")
# except ConnectionError:
#     print("All retries exhausted")

---

## Summary

### Error Handling Best Practices

| Do | Do Not |
|----|--------|
| Catch specific exceptions | Use bare `except:` |
| Create custom exceptions with context | Raise generic `Exception` |
| Use `finally` for cleanup | Leave files/connections open |
| Validate inputs early with clear messages | Let cryptic errors propagate |
| Use logging in production | Scatter print() everywhere |
| Use EAFP pattern | Over-check with LBYL |

### Debugging Strategies (in order of complexity)

1. **Read the traceback** -- Python tells you exactly where and what
2. **Print debugging** -- quick, toggle with a flag
3. **Assertions** -- document and check invariants
4. **`breakpoint()` / `pdb`** -- interactive stepping for complex bugs
5. **Logging** -- persistent, configurable, production-ready

In [ ]:
# Cleanup
import os
for f in ['test.fasta', 'bad.fasta', 'bad2.fasta']:
    if os.path.exists(f):
        os.remove(f)
        print(f"Cleaned up: {f}")

---

[< Previous: Decorators and Context Managers](../14_Decorators_and_Context_Managers/02_context_managers.ipynb) | [Tier 1: Python for Bioinformatics Overview](../README.md) | [Next: Numpy >](../16_NumPy_and_Pandas/01_numpy.ipynb)

---